In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

SEEDS = [42, 52, 62, 72, 82]

BEST_UNITS = 16
BEST_DROPOUT = 0.3
BEST_BATCH = 512

FREEZE_EPOCHS = 5
FINETUNE_EPOCHS = 30

print("TensorFlow version:", tf.__version__)
print("SEEDS:", SEEDS)

2026-04-07 16:06:51.711802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775578011.929727      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775578012.002153      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775578012.536532      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775578012.536582      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775578012.536585      17 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
SEEDS: [42, 52, 62, 72, 82]


In [3]:
INPUT_PATH_OLD = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val"

X_train = np.load(os.path.join(INPUT_PATH_OLD, "X_train.npy")).astype(np.float32)
X_val   = np.load(os.path.join(INPUT_PATH_OLD, "X_val.npy")).astype(np.float32)
X_test  = np.load(os.path.join(INPUT_PATH_OLD, "X_test.npy")).astype(np.float32)

y_train = np.load(os.path.join(INPUT_PATH_OLD, "y_train.npy")).astype(np.float32).reshape(-1)
y_val   = np.load(os.path.join(INPUT_PATH_OLD, "y_val.npy")).astype(np.float32).reshape(-1)
y_test  = np.load(os.path.join(INPUT_PATH_OLD, "y_test.npy")).astype(np.float32).reshape(-1)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)

X_train: (145772, 10, 133)
X_val: (36597, 10, 133)
X_test: (45552, 10, 133)
y_train: (145772,)


In [4]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, class_weights)}
print("Class weights:", class_weights_dict)

Class weights: {0: 0.5407133742841034, 1: 6.64048833819242}


In [5]:
PRETRAINED_ENCODER_PATH = "/kaggle/input/datasets/thuhiuhong/pretrain-lstm/encoder_pretrained_vital.weights.h5"

print("Exists:", os.path.exists(PRETRAINED_ENCODER_PATH))

Exists: True


In [6]:
def set_all_seeds(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [7]:
def build_encoder(n_lstm_units, dropout_rate, input_shape):
    inputs = Input(shape=input_shape, name='encoder_input')

    x = Bidirectional(
        LSTM(n_lstm_units, return_sequences=True),
        name='bilstm_1'
    )(inputs)
    x = Dropout(dropout_rate, name='dropout_1')(x)
    x = BatchNormalization(name='bn_1')(x)

    x = Bidirectional(
        LSTM(max(n_lstm_units // 2, 8), return_sequences=False),
        name='bilstm_2'
    )(x)
    x = Dropout(dropout_rate, name='dropout_2')(x)
    x = BatchNormalization(name='bn_2')(x)

    shared = Dense(16, activation='relu', name='shared_dense')(x)

    encoder = Model(inputs, shared, name='encoder')
    return encoder

In [8]:
def build_sepsis_model_from_encoder(n_lstm_units, dropout_rate, input_shape):
    encoder = build_encoder(n_lstm_units, dropout_rate, input_shape)

    inputs = encoder.input
    shared = encoder.output

    x = Dropout(0.2, name='head_dropout')(shared)
    sepsis_out = Dense(1, activation='sigmoid', name='sepsis')(x)

    model = Model(inputs, sepsis_out, name='sepsis_finetune_model')
    return model, encoder

In [9]:
def train_one_pretrained_member(seed, member_idx):
    print(f"\n===== TRAIN MEMBER {member_idx} | seed={seed} =====")

    K.clear_session()
    set_all_seeds(seed)

    model, encoder = build_sepsis_model_from_encoder(
        n_lstm_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT,
        input_shape=(X_train.shape[1], X_train.shape[2])
    )

    encoder.load_weights(PRETRAINED_ENCODER_PATH)
    print("Loaded pretrained encoder weights.")

    # ===== Stage A: freeze encoder =====
    for layer in encoder.layers:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )

    history_stageA = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=FREEZE_EPOCHS,
        batch_size=BEST_BATCH,
        class_weight=class_weights_dict,
        verbose=1
    )

    # ===== Stage B: unfreeze encoder =====
    for layer in encoder.layers:
        layer.trainable = True

    ckpt_path = f"/kaggle/working/member_{member_idx}_best.keras"

    early_stopping = EarlyStopping(
        monitor='val_auprc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_auprc',
        mode='max',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )

    checkpoint = ModelCheckpoint(
        ckpt_path,
        monitor='val_auprc',
        mode='max',
        save_best_only=True,
        verbose=1
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )

    history_stageB = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=FINETUNE_EPOCHS,
        batch_size=BEST_BATCH,
        class_weight=class_weights_dict,
        callbacks=[early_stopping, reduce_lr, checkpoint],
        verbose=1
    )

    model.load_weights(ckpt_path)

    val_prob = model.predict(X_val, batch_size=BEST_BATCH, verbose=0).ravel()
    test_prob = model.predict(X_test, batch_size=BEST_BATCH, verbose=0).ravel()

    val_auroc = roc_auc_score(y_val, val_prob)
    val_auprc = average_precision_score(y_val, val_prob)

    print(f"Member {member_idx} val AUROC: {val_auroc:.4f}")
    print(f"Member {member_idx} val AUPRC: {val_auprc:.4f}")

    return {
        "model": model,
        "val_prob": val_prob,
        "test_prob": test_prob,
        "history_stageA": history_stageA.history,
        "history_stageB": history_stageB.history,
        "val_auroc": val_auroc,
        "val_auprc": val_auprc
    }

In [10]:
member_results = []

for i, seed in enumerate(SEEDS, start=1):
    result = train_one_pretrained_member(seed=seed, member_idx=i)
    member_results.append(result)


===== TRAIN MEMBER 1 | seed=42 =====


2026-04-07 16:07:32.837813: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Loaded pretrained encoder weights.
Epoch 1/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - auprc: 0.0787 - auroc: 0.5027 - loss: 0.7986 - precision: 0.0766 - recall: 0.3169 - val_auprc: 0.0937 - val_auroc: 0.5397 - val_loss: 0.6960 - val_precision: 0.0855 - val_recall: 0.4902
Epoch 2/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - auprc: 0.0947 - auroc: 0.5402 - loss: 0.7143 - precision: 0.0838 - recall: 0.5250 - val_auprc: 0.1157 - val_auroc: 0.6036 - val_loss: 0.6789 - val_precision: 0.1039 - val_recall: 0.5636
Epoch 3/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - auprc: 0.1079 - auroc: 0.5735 - loss: 0.6936 - precision: 0.0912 - recall: 0.5420 - val_auprc: 0.1243 - val_auroc: 0.6276 - val_loss: 0.6717 - val_precision: 0.1088 - val_recall: 0.5736
Epoch 4/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - auprc: 0.1180 - auroc: 0.5900 - loss: 0.6843 - precision: 0.0961 - recall: 0.5511 - val_auprc: 0.1277 - val_auroc: 0.6375 - val_loss: 0.6626 - val_precision: 0.1111 - val_recall: 0.5525
Epoc

In [11]:
val_probs_members = np.column_stack([r["val_prob"] for r in member_results])
test_probs_members = np.column_stack([r["test_prob"] for r in member_results])

val_prob_ens = val_probs_members.mean(axis=1)
test_prob_ens = test_probs_members.mean(axis=1)

print("val_probs_members:", val_probs_members.shape)
print("test_probs_members:", test_probs_members.shape)
print("val_prob_ens:", val_prob_ens.shape)
print("test_prob_ens:", test_prob_ens.shape)

val_probs_members: (36597, 5)
test_probs_members: (45552, 5)
val_prob_ens: (36597,)
test_prob_ens: (45552,)


In [12]:
val_auroc_ens = roc_auc_score(y_val, val_prob_ens)
val_auprc_ens = average_precision_score(y_val, val_prob_ens)

test_auroc_ens = roc_auc_score(y_test, test_prob_ens)
test_auprc_ens = average_precision_score(y_test, test_prob_ens)

print("=== ENSEMBLE THRESHOLD-INDEPENDENT METRICS ===")
print(f"Validation AUROC: {val_auroc_ens:.4f}")
print(f"Validation AUPRC: {val_auprc_ens:.4f}")
print(f"Test AUROC:       {test_auroc_ens:.4f}")
print(f"Test AUPRC:       {test_auprc_ens:.4f}")

=== ENSEMBLE THRESHOLD-INDEPENDENT METRICS ===
Validation AUROC: 0.8362
Validation AUPRC: 0.3424
Test AUROC:       0.8579
Test AUPRC:       0.3641


In [13]:
rows = []
for th in np.arange(0.05, 0.96, 0.01):
    val_pred = (val_prob_ens >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, val_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    youden_j = sensitivity + specificity - 1

    rows.append({
        "threshold": round(th, 2),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "youden_j": youden_j
    })

df_th = pd.DataFrame(rows)

candidate = df_th[df_th["sensitivity"] >= 0.80].copy()
best = candidate.sort_values("specificity", ascending=False).iloc[0]

print(best)

threshold      0.220000
sensitivity    0.805227
specificity    0.722074
youden_j       0.527302
Name: 17, dtype: float64


In [14]:
best_threshold = float(best["threshold"])
test_pred = (test_prob_ens >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
acc = accuracy_score(y_test, test_pred)
youden_j = sensitivity + specificity - 1

auroc = roc_auc_score(y_test, test_prob_ens)
auprc = average_precision_score(y_test, test_prob_ens)

print(f"=== PRETRAINED SIMPLE ENSEMBLE | THRESHOLD = {best_threshold:.2f} ===")
print(f"AUROC:       {auroc:.4f}")
print(f"AUPRC:       {auprc:.4f}")
print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Youden's J:  {youden_j:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

=== PRETRAINED SIMPLE ENSEMBLE | THRESHOLD = 0.22 ===
AUROC:       0.8579
AUPRC:       0.3641
Accuracy:    0.7342
Sensitivity: 0.8323
Specificity: 0.7268
Precision:   0.1879
Recall:      0.8323
F1-score:    0.3066
Youden's J:  0.5591

Confusion Matrix:
[[30770 11567]
 [  539  2676]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9828    0.7268    0.8356     42337
         1.0     0.1879    0.8323    0.3066      3215

    accuracy                         0.7342     45552
   macro avg     0.5853    0.7796    0.5711     45552
weighted avg     0.9267    0.7342    0.7983     45552

